# Assignment 5 — CNNs & Neural Style Transfer

**Task 1**: CNN from scratch on the Seeds dataset (Regression)

**Task 2**: Neural Style Transfer + Human Matting + Video Pipeline

---

## 0. Environment setup

In [ ]:
# Install dependencies (run once)
# !pip install torch torchvision tensorboard scikit-learn matplotlib opencv-python-headless pyyaml pillow


In [ ]:
import sys, os, random, json, csv
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"cuda" if torch.cuda.is_available() else "cpu"}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

---
# TASK 1 — CNN From Scratch (Seeds Dataset)

## 1.1 Dataset exploration

In [ ]:
sys.path.insert(0, 'task1_cnn')
from task1_cnn.data import load_config, seed_everything, build_dataloaders

cfg = load_config('task1_cnn/config.yaml')
seed_everything(cfg['random_seed'])

train_loader, val_loader, test_loader, n_out = build_dataloaders(cfg)
print(f'Outputs : {n_out}')
print(f'Train   : {len(train_loader.dataset)}')
print(f'Val     : {len(val_loader.dataset)}')
print(f'Test    : {len(test_loader.dataset)}')

In [ ]:
# Visualise a batch of training images
MEAN = torch.tensor([0.485, 0.456, 0.406])
STD  = torch.tensor([0.229, 0.224, 0.225])

imgs, labels = next(iter(train_loader))
n_show = min(16, len(imgs))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes = axes.flatten()
for i in range(n_show):
    img = (imgs[i] * STD.view(3,1,1) + MEAN.view(3,1,1)).clamp(0,1).permute(1,2,0).numpy()
    axes[i].imshow(img)
    axes[i].set_title(f'NormVal {labels[i].item():.2f}', fontsize=7)
    axes[i].axis('off')
plt.suptitle('Training batch (augmented)', fontsize=12)
plt.tight_layout()
plt.show()

## 1.2 Model architectures

In [ ]:
from task1_cnn.models import BaselineCNN, DeepRegCNN, build_model

model_a = BaselineCNN(num_outputs=1)
model_b = DeepRegCNN(num_outputs=1, dropout=0.3, residual=True)

dummy = torch.randn(2, 3, 128, 128)
print('Model A — BaselineCNN')
print(f'  Output: {model_a(dummy).shape}')
print(f'  Params: {model_a.count_parameters():,}  (must be ≤ 1,500,000)')
assert model_a.count_parameters() <= 1_500_000, 'Model A exceeds 1.5M parameters!'

print("\\nModel B — DeepRegCNN")
print(f'  Output: {model_b(dummy).shape}')
print(f'  Params: {model_b.count_parameters():,}')

## 1.3 Optimizer exploration — 6 configurations on Model A

In [ ]:
import importlib, task1_cnn.train as tr_mod
importlib.reload(tr_mod)

# Shorten epochs for notebook demo
cfg_demo = load_config('task1_cnn/config.yaml')
cfg_demo['training']['epochs'] = 5
cfg_demo['training']['early_stopping_patience'] = 3

opts = ['adam_fast', 'adam_standard', 'adam_stable', 'sgd_fast', 'sgd_standard', 'sgd_stable']
results_sweep = []
for opt in opts:
    print(f'Training with {opt}...')
    res = tr_mod.train(cfg_demo, model_key='model_a', opt_override=opt)
    results_sweep.append(res)

for r in results_sweep:
    print(f"{r['optimizer']:15s} | Test MSE: {r['test_mse']:.6f} | MAE: {r['test_mae']:.6f}")

In [ ]:
# Plot optimizer comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for r in results_sweep:
    h = r['history']
    label = r['optimizer']
    epochs = range(1, len(h['val_loss']) + 1)
    ax1.plot(epochs, h['val_loss'], label=label)
    ax2.plot(epochs, h['val_mae'],  label=label)

ax1.set_title('Validation MSE'); ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7)
ax2.set_title('Validation MAE'); ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7)
plt.suptitle('Optimizer Exploration — Model A', fontsize=13)
plt.tight_layout()
plt.savefig('task1_cnn/cnn_outputs_reg/optimizer_comparison.png', dpi=100)
plt.show()

## 1.4 Train Model B (deeper + regularised)

In [ ]:
r_b = tr_mod.train(cfg_demo, model_key='model_b', opt_override='adam_standard')
print(f'Model B test MSE: {r_b["test_mse"]:.6f}')

## 1.5 Full evaluation — metrics + confusion matrix + failure cases

In [ ]:
sys.path.insert(0, 'task1_cnn')
from task1_cnn.data import load_config, seed_everything, build_dataloaders

cfg = load_config('task1_cnn/config.yaml')
seed_everything(cfg['random_seed'])

train_loader, val_loader, test_loader, n_out = build_dataloaders(cfg)
print(f'Outputs : {n_out}')
print(f'Train   : {len(train_loader.dataset)}')
print(f'Val     : {len(val_loader.dataset)}')
print(f'Test    : {len(test_loader.dataset)}')

In [ ]:
# Regression Scatter Plot for Model A
wt = Path('task1_cnn/weights_regression/BaselineCNN_adam_standard_best.pth')
if wt.exists():
    model_a_eval = build_model(cfg, 'model_a', n_out).to(device)
    model_a_eval.load_state_dict(torch.load(wt, map_location=device))
    y_true, y_pred = ev_mod.get_predictions(model_a_eval, test_loader, device)
    
    ev_mod.plot_regression_scatter(y_true, y_pred, 'task1_cnn/cnn_outputs_reg/scatter_A.png')
    plt.figure(figsize=(8, 8))
    plt.imshow(plt.imread('task1_cnn/cnn_outputs_reg/scatter_A.png'))
    plt.axis('off'); plt.title('Model A — Predicted vs True Scatter', fontsize=13)
    plt.show()

In [ ]:
rows = []
for name, m in results.items():
    rows.append({
        'Method':              name,
        'MSE':                 f"{m['mse']:.6f}",
        'MAE':                 f"{m['mae']:.6f}",
    })

build_comparison_table(rows, 'task1_cnn/cnn_outputs_reg/comparison_table.csv')

---
# TASK 2 — Neural Style Transfer + Human Matting + Video

## 2.1 NST sanity check — single transfer

In [ ]:
sys.path.insert(0, 'task2_nst_video')
from task2_nst_video.nst import run_nst, load_image, tensor_to_pil

# ── Make sure you have at least one content and one style image ──
CONTENT = 'task2_nst_video/content/frame_01.jpg'
STYLE   = 'task2_nst_video/style/starry_night.jpg'
OUT     = 'task2_nst_video/outputs/test_stylized.png'

if Path(CONTENT).exists() and Path(STYLE).exists():
    Path('task2_nst_video/outputs').mkdir(parents=True, exist_ok=True)
    result = run_nst(CONTENT, STYLE, OUT, beta=1e5, n_steps=200)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, path, title in zip(axes,
                                [CONTENT, STYLE, OUT],
                                ['Content', 'Style', 'Stylized']):
        ax.imshow(plt.imread(path))
        ax.set_title(title); ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('Place content/frame_01.jpg and style/starry_night.jpg first.')

## 2.2 β/α ratio sweep

In [ ]:
from task2_nst_video.nst import sweep_beta_alpha

if Path(CONTENT).exists() and Path(STYLE).exists():
    sweep_beta_alpha(CONTENT, STYLE,
                    out_dir='task2_nst_video/outputs',
                    ratios=(1e3, 1e5, 1e7))
    plt.figure(figsize=(15, 5))
    plt.imshow(plt.imread('task2_nst_video/outputs/beta_alpha_ablation.png'))
    plt.axis('off'); plt.title('β/α Ablation', fontsize=14)
    plt.tight_layout(); plt.show()

## 2.3 Layer ablation — shallow vs deep style layers

In [ ]:
from task2_nst_video.nst import layer_ablation

if Path(CONTENT).exists() and Path(STYLE).exists():
    layer_ablation(CONTENT, STYLE, out_dir='task2_nst_video/outputs')
    plt.figure(figsize=(15, 5))
    plt.imshow(plt.imread('task2_nst_video/outputs/layer_ablation.png'))
    plt.axis('off'); plt.title('Layer Ablation', fontsize=14)
    plt.tight_layout(); plt.show()

print("""
Layer Ablation Discussion:
  • Shallow layers (relu1_1, relu2_1) capture fine textures — brushstroke patterns,
    colour palettes, and local frequency patterns dominate.
  • Deep layers (relu4_1, relu5_1) capture higher-level structural style — swirling
    shapes and large compositional patterns appear more clearly.
  • Using all five layers gives the best balance of local texture and global structure.
""")

## 2.4 5×3 NST grid

In [ ]:
from task2_nst_video.nst import build_grid

content_dir = 'task2_nst_video/content'
style_dir   = 'task2_nst_video/style'
n_content   = len(list(Path(content_dir).glob('*.jpg')))
n_style     = len([p for p in Path(style_dir).glob('*')
                   if p.suffix.lower() in ('.jpg','.jpeg','.png')])

if n_content >= 5 and n_style >= 3:
    build_grid(content_dir, style_dir, 'task2_nst_video/outputs')
    plt.figure(figsize=(12, 16))
    plt.imshow(plt.imread('task2_nst_video/outputs/grid.png'))
    plt.axis('off'); plt.title('5 Content × 3 Style Grid', fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print(f'Need 5 content images ({n_content} found) and 3 style images ({n_style} found).')

## 2.5 Train the matting model

In [ ]:
# Run from terminal:
#   python task2_nst_video/matting/train.py \
#       --data   data/aisegment \
#       --epochs 30 \
#       --out    task2_nst_video/matting/weights

# Quick smoke-test of the model architecture:
from task2_nst_video.matting.model import MattingUNet
import torch

m = MattingUNet(pretrained=False)
dummy = torch.randn(1, 3, 256, 256)
alpha = m(dummy)
print(f'MattingUNet output: {alpha.shape}  range [{alpha.min():.3f}, {alpha.max():.3f}]')
assert alpha.shape == (1, 1, 256, 256)
print('Architecture OK')

In [ ]:
# Plot matting training curves (after training is complete)
log_path = 'task2_nst_video/matting/weights/matting_log.csv'
if Path(log_path).exists():
    epochs, tr_loss, va_loss, va_iou = [], [], [], []
    with open(log_path) as f:
        for row in csv.DictReader(f):
            epochs.append(int(row['epoch']))
            tr_loss.append(float(row['train_loss']))
            va_loss.append(float(row['val_loss']))
            va_iou.append(float(row['val_iou']))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, tr_loss, label='Train'); ax1.plot(epochs, va_loss, label='Val')
    ax1.set_title('Matting Loss (L1 + Dice)'); ax1.legend()
    ax2.plot(epochs, va_iou, color='tab:green')
    ax2.axhline(0.85, ls='--', color='red', label='Target IoU=0.85')
    ax2.set_title('Validation IoU'); ax2.legend()
    plt.suptitle('Human Matting Training', fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print('Train the matting model first.')

## 2.6 Matting visualisation on video frames

In [ ]:
from task2_nst_video.matting.model import MattingUNet
import torchvision.transforms.functional as TF

matting_weights = 'task2_nst_video/matting/weights/matting_best.pth'
if Path(matting_weights).exists():
    mat_model = MattingUNet(pretrained=False).to(device)
    mat_model.load_state_dict(torch.load(matting_weights, map_location=device))
    mat_model.eval()

    frames_to_show = sorted(Path('task2_nst_video/content').glob('*.jpg'))[:5]
    fig, axes = plt.subplots(len(frames_to_show), 3, figsize=(9, len(frames_to_show)*3))
    if len(frames_to_show) == 1:
        axes = axes[np.newaxis, :]

    for i, fp in enumerate(frames_to_show):
        from PIL import Image as PILImage
        img = PILImage.open(fp).convert('RGB').resize((256, 256))
        t   = TF.to_tensor(img)
        t   = TF.normalize(t, [0.485,0.456,0.406],[0.229,0.224,0.225]).unsqueeze(0).to(device)
        with torch.no_grad():
            alpha = mat_model(t).squeeze().cpu().numpy()

        orig = np.array(img)
        cutout = orig.copy()
        cutout[alpha < 0.5] = 0

        axes[i][0].imshow(orig);   axes[i][0].set_title('Frame');  axes[i][0].axis('off')
        axes[i][1].imshow(alpha, cmap='gray'); axes[i][1].set_title('Alpha'); axes[i][1].axis('off')
        axes[i][2].imshow(cutout); axes[i][2].set_title('Cutout'); axes[i][2].axis('off')

    plt.suptitle('Human Matting — Sample Frames', fontsize=12)
    plt.tight_layout()
    plt.savefig('task2_nst_video/outputs/matting_overlay.png', dpi=100)
    plt.show()
else:
    print('Train the matting model first.')

## 2.7 Feature-map visualisation (VGG19)

In [ ]:
from task2_nst_video.nst import visualise_feature_maps

# Compare a video frame with a seed image
frame_img = 'task2_nst_video/content/frame_01.jpg'
seed_img  = 'task1_cnn/../seeds/1.jpg'  # adjust path

for img_path, tag in [(frame_img, 'video_frame'), (seed_img, 'seed_image')]:
    out = f'task2_nst_video/outputs/feature_maps_{tag}.png'
    if Path(img_path).exists():
        visualise_feature_maps(img_path, out)
        fig = plt.figure(figsize=(16, 4))
        plt.imshow(plt.imread(out))
        plt.axis('off')
        plt.title(f'Feature maps — {tag}', fontsize=12)
        plt.tight_layout(); plt.show()

print("""
Feature-map discussion:
  • Shallow VGG19 channels (relu1_1) respond to edges, colours, and simple textures —
    the same low-level features the first conv layers of our Task-1 CNN are learning.
  • Deeper channels (relu4_1) encode semantic object parts: leaf shape, vein patterns,
    and seed morphology — matching what the GAP + classifier head of Model B must learn.
""")

## 2.8 Run the full video pipeline

In [ ]:
# ── Step 1: extract content frames from your video ──────────
video_path = 'task2_nst_video/input_video.mp4'

if Path(video_path).exists():
    import subprocess
    subprocess.run([
        'python', 'task2_nst_video/video_pipeline.py',
        '--extract_frames',
        '--video', video_path,
        '--n_frames', '5'
    ], check=True)
else:
    print('Place input_video.mp4 in task2_nst_video/ first.')

In [ ]:
# ── Step 2: run the full stylization pipeline ────────────────
# NOTE: This is compute-intensive. Run on GPU or overnight on CPU.
# Adjust --nst_steps down (e.g. 100) for a quick test.

if Path(video_path).exists() and Path(matting_weights).exists():
    import subprocess
    subprocess.run([
        'python', 'task2_nst_video/video_pipeline.py',
        '--video',     video_path,
        '--style',     'task2_nst_video/style/starry_night.jpg',
        '--matting',   matting_weights,
        '--out_dir',   'task2_nst_video/outputs',
        '--nst_steps', '300',
        '--beta',      '1e5',
    ], check=True)
    print('Pipeline complete!')
else:
    print('Need input_video.mp4 and trained matting weights.')

## 2.9 Display output video frames

In [ ]:
import cv2

def show_video_thumbnail(video_path, title, frame_idx=0):
    if not Path(video_path).exists():
        print(f'  [skip] {video_path} not found')
        return None
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

variants = [
    ('task2_nst_video/outputs/stylized_background.mp4', 'Variant 1: BG Stylized'),
    ('task2_nst_video/outputs/stylized_subject.mp4',    'Variant 2: Subject Stylized'),
    ('task2_nst_video/outputs/stylized_full.mp4',       'Variant 3: Full Stylized'),
]

frames = [(show_video_thumbnail(p, t, 30), t) for p, t in variants]
frames = [(f, t) for f, t in frames if f is not None]

if frames:
    fig, axes = plt.subplots(1, len(frames), figsize=(5*len(frames), 5))
    if len(frames) == 1:
        axes = [axes]
    for ax, (frame, title) in zip(axes, frames):
        ax.imshow(frame); ax.set_title(title, fontsize=10); ax.axis('off')
    plt.suptitle('Stylized Video Variants (frame 30)', fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print('Run the video pipeline first.')

---
# Cross-method Summary

In [ ]:
# Read and display the final comparison table
table_path = 'task1_cnn/cnn_outputs_reg/comparison_table.csv'
if Path(table_path).exists():
    import pandas as pd
    df = pd.read_csv(table_path)
    display(df)
else:
    print('Run evaluation in 1.5 first.')

In [ ]:
print("""
Summary:
  Task 1 has been transitioned to a regression problem (predicting seed index).
  CNN Model A & B now output a single normalized scalar [0, 1].
  Optimizer exploration showed that different variants of Adam/SGD affect convergence speed for regression.

Deployment recommendation for AgriVision:
  • Seed Regression: Model B (DeepRegCNN) provides robust results; use normalized MSE as primary metric.
  • Video stylization: NST + matting pipeline is NOT real-time (L-BFGS is iterative).
    For production, consider fast-NST (Johnson et al., feed-forward network) combined
    with the trained matting model, enabling ~30 fps on a single GPU.
""")